# Hybrid KEP Model Reuse Benchmark

Goal: compare two ways to solve the same KEP graph under many random objective vectors.

1. Rebuild every time: construct a new Gurobi model for each edge-weight vector.
2. Reuse model: build the Gurobi model once, then only update objective coefficients.

The graph is randomly generated with 50 nodes. Edge weights are random integers in [0, 100].

In [ ]:
!pip -q install gurobipy numpy pandas matplotlib

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import gurobipy as gp
from gurobipy import GRB

NUM_NODES = 100
NUM_EDGES = 600
NUM_NDDS = 5
NUM_WEIGHTS = 100
MAX_CHAIN = 4
SEED = 42

In [ ]:
def random_graph(num_nodes=NUM_NODES, num_edges=NUM_EDGES, num_ndds=NUM_NDDS, seed=SEED):
    rng = np.random.default_rng(seed)
    edges = set()
    while len(edges) < num_edges:
        src = int(rng.integers(0, num_nodes))
        dst = int(rng.integers(num_ndds, num_nodes))
        if src != dst:
            edges.add((src, dst))
    edge_index = np.array(sorted(edges), dtype=np.int64).T
    is_ndd = np.zeros(num_nodes, dtype=bool)
    is_ndd[:num_ndds] = True
    return edge_index, is_ndd

def random_weights(num_vectors, num_edges, seed=SEED + 1):
    return np.random.default_rng(seed).integers(0, 101, size=(num_vectors, num_edges), dtype=np.int32)

edge_index, is_ndd = random_graph(NUM_NODES, NUM_EDGES, NUM_NDDS, SEED)
weights = random_weights(NUM_WEIGHTS, edge_index.shape[1], SEED + 1)
print(edge_index.shape, weights.shape)

(2, 600) (100, 600)


## Minimal Hybrid Chain Model Builder

This cell defines the reusable model-building function.

For the Colab demo, we include a simplified hybrid model with chains and cycles of length <= 3.
The main purpose of this benchmark is to demonstrate objective reuse: build the Gurobi model once, then repeatedly update only the objective.

In [ ]:
def build_edge_lists(edge_index, num_nodes):
    edge_index_np = np.asarray(edge_index)
    src = edge_index_np[0].astype(int)
    dst = edge_index_np[1].astype(int)

    outgoing = [[] for _ in range(num_nodes)]
    incoming = [[] for _ in range(num_nodes)]

    for e, (u, v) in enumerate(zip(src, dst)):
        outgoing[int(u)].append(e)
        incoming[int(v)].append(e)

    return src, dst, outgoing, incoming


def build_pief_chain_keys(src, dst, outgoing, is_ndd, max_chain):
    """
    PIEF-chain idea:
    position 1 can only start from NDDs.
    position p can only start from nodes reachable at position p-1.
    This avoids creating many impossible edge-position variables.
    """
    keys = []

    active_donors = {
        node for node in range(len(outgoing))
        if bool(is_ndd[node])
    }

    for pos in range(1, max_chain + 1):
        next_active = set()

        for donor in active_donors:
            for e in outgoing[donor]:
                recipient = int(dst[e])

                # chains should not go into NDD nodes
                if bool(is_ndd[recipient]):
                    continue

                keys.append((e, pos))
                next_active.add(recipient)

        active_donors = next_active

        if not active_donors:
            break

    return keys


def find_cycle_candidates(edge_index, is_ndd, num_nodes, max_cycle=3):
    """
    CF-cycle part:
    enumerate cycle candidates first, then create one binary variable per cycle.
    Each candidate is a dict with nodes and edges.
    """
    src, dst = edge_index
    adj = {i: [] for i in range(num_nodes)}

    for e, (s, d) in enumerate(zip(src, dst)):
        s = int(s)
        d = int(d)
        adj[s].append((d, e))

    cycle_edge_tuples = []

    for start in range(num_nodes):
        if bool(is_ndd[start]):
            continue

        stack = [(start, [start], [])]

        while stack:
            curr, path_nodes, path_edges = stack.pop()

            if len(path_edges) >= max_cycle:
                continue

            for nxt, e in adj[curr]:
                if bool(is_ndd[nxt]):
                    continue

                new_edges = path_edges + [e]

                if nxt == start and len(new_edges) >= 2:
                    cycle_edge_tuples.append(tuple(new_edges))

                elif nxt not in path_nodes and nxt > start:
                    stack.append((nxt, path_nodes + [nxt], new_edges))

    cycle_edge_tuples = sorted(set(cycle_edge_tuples))

    candidates = []
    for edges in cycle_edge_tuples:
        nodes = [int(src[e]) for e in edges]
        candidates.append({
            "type": "cycle",
            "nodes": nodes,
            "edges": list(edges),
        })

    return candidates


def build_model(edge_index, is_ndd, num_nodes, max_chain=MAX_CHAIN, max_cycle=3, env=None):
    src, dst, outgoing, incoming = build_edge_lists(edge_index, num_nodes)

    cycle_candidates = find_cycle_candidates(
        edge_index=edge_index,
        is_ndd=is_ndd,
        num_nodes=num_nodes,
        max_cycle=max_cycle,
    )

    valid_chain_keys = build_pief_chain_keys(
        src=src,
        dst=dst,
        outgoing=outgoing,
        is_ndd=is_ndd,
        max_chain=max_chain,
    )

    chain_incoming = {}
    chain_outgoing = {}

    for e, pos in valid_chain_keys:
        chain_incoming.setdefault((int(dst[e]), pos), []).append((e, pos))
        chain_outgoing.setdefault((int(src[e]), pos), []).append((e, pos))

    model = gp.Model("cf_cycle_pief_chain", env=env)
    model.Params.OutputFlag = 0
    model.ModelSense = GRB.MAXIMIZE

    cycle_vars = model.addVars(len(cycle_candidates), vtype=GRB.BINARY, name="cycle")
    chain_vars = model.addVars(valid_chain_keys, vtype=GRB.BINARY, name="chain")

    pair_nodes = [i for i in range(num_nodes) if not bool(is_ndd[i])]
    ndd_nodes = [i for i in range(num_nodes) if bool(is_ndd[i])]

    # Save objective variable groups.
    # A cycle variable contributes sum of its edge weights.
    # A chain variable contributes its own edge weight.
    obj_vars = []
    obj_edge_groups = []

    for c_idx, candidate in enumerate(cycle_candidates):
        obj_vars.append(cycle_vars[c_idx])
        obj_edge_groups.append(tuple(candidate["edges"]))

    for e, pos in valid_chain_keys:
        obj_vars.append(chain_vars[e, pos])
        obj_edge_groups.append((e,))

    # each pair node can be used at most once: either in one cycle or as recipient in one chain
    for node in pair_nodes:
        cycle_usage = gp.quicksum(
            cycle_vars[c_idx]
            for c_idx, candidate in enumerate(cycle_candidates)
            if node in candidate["nodes"]
        )

        chain_usage = gp.quicksum(
            chain_vars[e, pos]
            for e, pos in valid_chain_keys
            if int(dst[e]) == node
        )

        model.addConstr(cycle_usage + chain_usage <= 1, name=f"pair_once_{node}")

    # each NDD starts at most one chain
    for node in ndd_nodes:
        model.addConstr(
            gp.quicksum(
                chain_vars[e, 1]
                for e, pos in valid_chain_keys
                if pos == 1 and int(src[e]) == node
            ) <= 1,
            name=f"ndd_once_{node}",
        )

    # PIEF chain flow:
    # a paired donor can donate at position p only if its paired recipient received at p-1
    for node in pair_nodes:
        for pos in range(2, max_chain + 1):
            outgoing_now = gp.quicksum(
                chain_vars[e, pos]
                for e, _ in chain_outgoing.get((node, pos), [])
            )

            incoming_prev = gp.quicksum(
                chain_vars[e, pos - 1]
                for e, _ in chain_incoming.get((node, pos - 1), [])
            )

            model.addConstr(
                outgoing_now <= incoming_prev,
                name=f"chain_flow_{node}_{pos}",
            )

    model.update()

    return {
        "model": model,
        "chain_vars": chain_vars,
        "cycle_vars": cycle_vars,
        "valid_chain_keys": valid_chain_keys,
        "cycle_candidates": cycle_candidates,
        "obj_vars": obj_vars,
        "obj_edge_groups": obj_edge_groups,
    }


def update_objective(model_data, w):
    """
    Faster objective update:
    directly change objective coefficients instead of rebuilding a LinExpr.
    """
    obj_values = [
        float(sum(w[e] for e in edge_group))
        for edge_group in model_data["obj_edge_groups"]
    ]

    model_data["model"].setAttr(
        GRB.Attr.Obj,
        model_data["obj_vars"],
        obj_values,
    )
    model_data["model"].update()

In [ ]:
src, dst, outgoing, incoming = build_edge_lists(edge_index, NUM_NODES)

cycle_candidates = find_cycle_candidates(
    edge_index=edge_index,
    is_ndd=is_ndd,
    num_nodes=NUM_NODES,
    max_cycle=3,
)

pief_chain_keys = build_pief_chain_keys(
    src=src,
    dst=dst,
    outgoing=outgoing,
    is_ndd=is_ndd,
    max_chain=MAX_CHAIN,
)

print(f"Found {len(cycle_candidates)} cycle candidates of length <= 3")
print(f"Found {len(pief_chain_keys)} PIEF chain variables")

Found 93 cycle candidates of length <= 3
Found 1272 PIEF chain variables


In [ ]:
def solve_rebuild(w, edge_index, is_ndd, num_nodes, max_chain, env):
    model_data = build_model(
        edge_index=edge_index,
        is_ndd=is_ndd,
        num_nodes=num_nodes,
        max_chain=max_chain,
        max_cycle=3,
        env=env,
    )

    update_objective(model_data, w)

    model = model_data["model"]
    model.optimize()

    if model.Status != GRB.OPTIMAL:
        raise RuntimeError(f"Rebuild model not optimal. Status={model.Status}")

    result = {
        "obj": float(model.ObjVal),
        "runtime": float(model.Runtime),      # Gurobi native optimize time
        "nodes": float(model.NodeCount),      # optional: B&B nodes
        "status": int(model.Status),
    }

    model.dispose()
    return result


class ReusableModel:
    def __init__(self, edge_index, is_ndd, num_nodes, max_chain, env):
        self.model_data = build_model(
            edge_index=edge_index,
            is_ndd=is_ndd,
            num_nodes=num_nodes,
            max_chain=max_chain,
            max_cycle=3,
            env=env,
        )

    def solve(self, w, reset_before_solve=False):
        update_objective(self.model_data, w)

        model = self.model_data["model"]

        if reset_before_solve:
            model.reset()

        model.optimize()

        if model.Status != GRB.OPTIMAL:
            raise RuntimeError(f"Reusable model not optimal. Status={model.Status}")

        return {
            "obj": float(model.ObjVal),
            "runtime": float(model.Runtime),   # Gurobi native optimize time
            "nodes": float(model.NodeCount),
            "status": int(model.Status),
        }

    def dispose(self):
        self.model_data["model"].dispose()


env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

# -------------------------
# Rebuild every time
# -------------------------
t0 = time.perf_counter()
rebuild_results = [
    solve_rebuild(w, edge_index, is_ndd, NUM_NODES, MAX_CHAIN, env)
    for w in weights
]
rebuild_wall_time = time.perf_counter() - t0

rebuild_obj = np.array([r["obj"] for r in rebuild_results])
rebuild_gurobi_time = sum(r["runtime"] for r in rebuild_results)
rebuild_nodes = sum(r["nodes"] for r in rebuild_results)


# -------------------------
# Reuse model
# -------------------------
t0 = time.perf_counter()
cached = ReusableModel(edge_index, is_ndd, NUM_NODES, MAX_CHAIN, env)
reuse_build_wall_time = time.perf_counter() - t0

t0 = time.perf_counter()
reuse_results = [
    cached.solve(w, reset_before_solve=False)
    for w in weights
]
reuse_wall_time = time.perf_counter() - t0

reuse_obj = np.array([r["obj"] for r in reuse_results])
reuse_gurobi_time = sum(r["runtime"] for r in reuse_results)
reuse_nodes = sum(r["nodes"] for r in reuse_results)


# -------------------------
# Check correctness
# -------------------------
diff = np.abs(rebuild_obj - reuse_obj)
max_diff = diff.max()
bad_idx = diff.argmax()

tol = 1e-5

print("=== Gurobi-native optimize time ===")
print(f"Rebuild optimize time : {rebuild_gurobi_time:.4f}s")
print(f"Reuse optimize time   : {reuse_gurobi_time:.4f}s")
print(f"Gurobi solve speedup  : {rebuild_gurobi_time / reuse_gurobi_time:.2f}x")
print()

print("=== Python wall-clock time ===")
print(f"Rebuild wall time     : {rebuild_wall_time:.4f}s")
print(f"Reuse build wall time : {reuse_build_wall_time:.4f}s")
print(f"Reuse solve wall time : {reuse_wall_time:.4f}s")
print(f"Wall-clock speedup    : {rebuild_wall_time / reuse_wall_time:.2f}x")
print()

print("=== Overhead estimate ===")
print(f"Rebuild overhead      : {rebuild_wall_time - rebuild_gurobi_time:.4f}s")
print(f"Reuse overhead        : {reuse_wall_time - reuse_gurobi_time:.4f}s")
print()

print("=== MIP search effort ===")
print(f"Rebuild total nodes   : {rebuild_nodes:.0f}")
print(f"Reuse total nodes     : {reuse_nodes:.0f}")
print()

print("=== Objective check ===")
print(f"Max obj diff          : {max_diff:.6g}")

if max_diff >= tol:
    print(f"Bad index             : {bad_idx}")
    print(f"Rebuild obj           : {rebuild_obj[bad_idx]}")
    print(f"Reuse obj             : {reuse_obj[bad_idx]}")
    raise AssertionError("Rebuild and reuse objectives are different beyond numerical tolerance.")
else:
    print(f"Objective check       : PASS, max diff <= {tol}")

cached.dispose()
env.dispose()

=== Gurobi-native optimize time ===
Rebuild optimize time : 13.7151s
Reuse optimize time   : 9.8868s
Gurobi solve speedup  : 1.39x

=== Python wall-clock time ===
Rebuild wall time     : 21.9479s
Reuse build wall time : 0.0488s
Reuse solve wall time : 10.1126s
Wall-clock speedup    : 2.17x

=== Overhead estimate ===
Rebuild overhead      : 8.2328s
Reuse overhead        : 0.2257s

=== MIP search effort ===
Rebuild total nodes   : 147
Reuse total nodes     : 142

=== Objective check ===
Max obj diff          : 0
Objective check       : PASS, max diff <= 1e-05
